# This notebook is a test play ground of how I will scrape the data and format to build the scraping script


In [1]:
import subprocess
import re
from fake_useragent import UserAgent
import undetected_chromedriver as uc
from  bs4 import BeautifulSoup
import requests
from curl_cffi import requests as curl_requests
import pandas as pd
import time 
import yaml



147


In [2]:
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

### SmartScrapper Class:
   This class is made to be able to scrape any webpage even after if the website detected that I am a bot<br> But how? the idea is simple there are two approaches: 
   1. The normal request I make a normal http request using curl_requessts where it helps me provide:
      1. a normaler header that contain more information the most important part the referer the previous link before this request
      2. Impersonate option it acts like a specific chromedriver 120 in my case 
      3. After the response I check if it's a challenge or not if it is a challenge I go to option 2.
   2. Here I use selenium. It's simply a library that actually opens google chrome from my computer and act type the link so it can't be detected but this approach it's only downside it's slower than the normal request approach so this is why it's used as a fallback strategy

In [3]:
# I have trouble scrapping the website using requests, so I will use a smarter approach 


class SmartScraper:
    def __init__(self, config):
        
        self.session = curl_requests.Session()
        self.driver = None
        self.config = config
    

    def get_chrome_version(self):
        try:
            output = subprocess.check_output(
                r'reg query "HKEY_CURRENT_USER\Software\Google\Chrome\BLBeacon" /v version',
                shell=True
            ).decode()
            version = re.search(r'[\d.]+', output).group()
            print(f"Detected Chrome version: {version}")
            return int(version)  # just the major version
        except:
            return None
    
    def format_url(self, slug):
        if '.com' in slug:
            return slug
        return self.config['data_source']['realstate_website']['base_url'] + slug
        
    def get_realistic_headers(self, referer=None):
        """Generate complete, realistic browser headers"""
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9,ar;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'same-origin' if referer else 'none',
            'Sec-Fetch-User': '?1',
            'sec-ch-ua': '"Not_A Brand";v="8", "Chromium";v="120"',
            'sec-ch-ua-mobile': '?0',
            'sec-ch-ua-platform': '"Windows"',
        }
        
        if referer:
            headers['Referer'] = referer
            
        return headers
        
    def try_requests_first(self, url, referer=None):
        """Try fast approach with realistic headers"""
        headers = self.get_realistic_headers(referer=referer)
        
        try:
            response = self.session.get(
                url, 
                headers=headers, 
                timeout=15,
                impersonate="chrome120"
                )
            print(f"Response status: {response.status_code}, length: {len(response.text)}")

            if (('humbucker' in response.text or
                'challenge' in response.text) and
                len(response.text) < 10000):
                return None
            if len(response.text) > 50000:
                print("Content looks valid, using requests response")
                return response.text
        
            return None
            
        except Exception as e:
            print(f"Request error: {e}")
            return None
    
    def use_selenium_fallback(self, url):
        """Fallback to Selenium if needed"""
        if not self.driver:
            version = self.get_chrome_version()
            options = uc.ChromeOptions()
            options.add_argument('--disable-blink-features=AutomationControlled')
            self.driver = uc.Chrome(options=options, version_main=version)
        
        self.driver.get(url)
        time.sleep(5)
        return self.driver.page_source
    
    def get_html(self, url, referer=None):
        """Smart approach: try fast method, fallback to slow"""
        print("Trying fast approach (requests)...")
        url = self.format_url(url)
        html = self.try_requests_first(url, referer=referer)
        
        if html:
            print("Success with requests!")
            return html
        
        print("Challenge detected. Using Selenium...")
        html = self.use_selenium_fallback(url)
        
        # Save cookies for next time
        if self.driver:
            cookies = self.driver.get_cookies()
            for cookie in cookies:
                self.session.cookies.set(
                    cookie['name'], 
                    cookie['value'],  
                    domain=cookie.get('domain', '')
                )
        
        return html
    def scrape_single_page(self, url):
        """Scrape a single page with smart approach"""
        html = self.get_html(url)
        
        soup = BeautifulSoup(html, 'lxml')
        # Your parsing code here
        
        return soup
    
    def scrape_multiple_pages(self, urls):
        """Scrape multiple pages efficiently"""
        results = []
        previous_url = None
        for i, url in enumerate(urls):
            print(f"\nPage {i+1}/{len(urls)}")
            html = self.get_html(url, referer=previous_url)
            
            soup = BeautifulSoup(html, 'lxml')
            # Your parsing code here
        
            results.append(soup)
            previous_url = self.format_url(url)
            time.sleep(2)  # Be nice to server
        
        return results
    
    def close(self):
        if self.driver:
            self.driver.quit()



In [4]:
def get_page_data(slug):
    scrapper = SmartScraper(config)
    try:
        
        page_soup = scrapper.scrape_single_page(slug)
       
    finally:    
        scrapper.close()
        print(f"Data extracted for {slug}")
        return page_soup
    
  

In [8]:
# get_page_data('apartments-duplex-for-sale/north-rehab-2/?page=5')

#### This is the main page that we are building our scrapping the location logic from 
##### The logic here we have 4 levels of locations:
1. City for now it will only be cairo
2. District like New Cairo, Zamelk, Nasr City
3. Areas each district may have more than on area like in new cairo there is 5th setlement, 1st setlment, ...etc
4. Neighborhood the lowest level <br>**Important note that there places like zamelk it doesn't have these sublevels or areas that doesn't have neihborhoods**<br>
so the following code will be to get this data and save it in the config file so I can fetch the data with structure afterward

In [6]:
soup = get_page_data('cairo')

Trying fast approach (requests)...
Response status: 200, length: 5549782
Content looks valid, using requests response
Success with requests!
Data extracted for cairo


In [14]:
# s = SmartScraper(config)
# x = s.format_url('some-page')
# print(x)



In [ ]:
# First we are going to construct our dataframe to hold our data evantullay 
# After some research I found how the data I will get will be structure
# The data will have the following fields: 
fields = config['data_source']['realstate_website']['fields']

In [24]:
print(f'Data fields to be collected: {len(fields)}')

Data fields to be collected: 33


In [4]:
# Here is a we will try to scrape a single listing page to test our selectors 
# So there are some information that we already know about the listing 
# 1. property type: apartment 2. sale/rent: sale 3. city: cairo 4. district: new cairo 5. area: fifth settlement 6. neighborhood: mountain view i city compound
# The third decision all urls will be typed in the config.yaml file which will not be shared publicly
url = config['data_source']['realstate_website']['base_url']+'cairo/'
headers = config['data_source']['realstate_website']['headers']
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

In [6]:
if 'humbucker' in soup.text:
    print("Challenge detected, need to use Selenium or a smarter approach")

In [26]:
# for now we will just get the data and put in a dictonary then we will make the appending logic later
page_data = {}
page_title = soup.select_one(config['data_source']['realstate_website']['selectors']['title']).text.strip()
page_data['title'] = page_title
page_data['price'] = soup.select_one(config['data_source']['realstate_website']['selectors']['price']).text.strip()
page_data['listing_date'] = soup.select_one(config['data_source']['realstate_website']['selectors']['creation_date']).text.strip()
page_data['seller_name'] = soup.select_one(config['data_source']['realstate_website']['selectors']['seller_name']).text.strip()
for key, value in page_data.items():
    print(f"{key}: {value}")


title: Apartment for sale 160M prime location lowest price in mountain view i city
price: EGP 6,000,000
listing_date: 1 week ago
seller_name: Views Real Estate


In [33]:
def extract_date(date):
    if 'today' in date.lower() or 'just now' in date.lower():
        listing_date = pd.Timestamp.now()
    elif 'yesterday' in date.lower():
        listing_date = pd.Timestamp.now() - pd.Timedelta(days=1)
    elif 'day' in date:
        days_ago = int(date.split()[0])
        listing_date = pd.Timestamp.now() - pd.Timedelta(days=days_ago)
    elif 'week' in date:
        weeks_ago = int(date.split()[0])
        listing_date = pd.Timestamp.now() - pd.Timedelta(weeks=weeks_ago)
    elif 'month' in date:
        months_ago = int(date.split()[0])
        listing_date = pd.Timestamp.now() - pd.Timedelta(days=30*months_ago)
    elif 'year' in date:
        years_ago = int(date.split()[0])
        listing_date = pd.Timestamp.now() - pd.Timedelta(days=365*years_ago)
    else:
        listing_date = None  # Handle unexpected format
    if listing_date:
        page_data['listing_date'] = listing_date.strftime('%m-%d-%Y')
    return listing_date.strftime('%m-%d-%Y'), type(listing_date) if listing_date else None

In [34]:
date = page_data['listing_date']

print(f"Listing date: {extract_date(date)}")

Listing date: ('01-08-2026', <class 'pandas._libs.tslibs.timestamps.Timestamp'>)


In [35]:
# Here comes the tricky part of extracting the data for the highlihts, details and features sections
# First we will extract the highlights section
def highlights_extractor(highlit_container, page_data):
    #   outer items contains 
    outer_items = highlit_container.select(config['data_source']['realstate_website']['selectors']['higlight_items'])

    for item in outer_items:
        spans = item.find_all('span')
        if len(spans) >=2:
            key = spans[0].text.strip().lower()
            value = spans[1].text.strip()
            page_data[key] = value
            print(f"Highlight - {key}: {value}")
    return page_data

In [36]:
highlights_container = soup.select_one(config['data_source']['realstate_website']['selectors']['highlights_container'])
page_data = highlights_extractor(highlights_container, page_data)
page_data['property_subtype'] = page_data.pop('type', None)

Highlight - type: Apartment
Highlight - ownership: Resale
Highlight - area (m²): 160
Highlight - bedrooms: 3
Highlight - bathrooms: 3
Highlight - furnished: No


In [37]:
details_container = soup.select_one('div[aria-label="Details"]')
# print(details_container.prettify())
details_itemss = details_container.select(config['data_source']['realstate_website']['selectors']['details_items'])
for item in details_itemss:
    spans = item.find_all('span')
    key = spans[0].text.strip().lower()
    value = spans[1].text.strip().lower()
    page_data[key] = value
    print(f"Detail - {key}: {value}")

Detail - level: 1
Detail - payment option: cash
Detail - completion status: ready


In [38]:
features_container = soup.select_one(config['data_source']['realstate_website']['selectors']['features_container'])
features_items = features_container.select(config['data_source']['realstate_website']['selectors']['features_items'])


In [39]:
for item in features_items:
    spans = item.find_all('span')
    for span in spans:
        key = span.text.strip().lower()
        if key == 'see all':
            continue
        
        page_data[key] = True
        print(f"Feature - {key}: True")

Feature - balcony: True
Feature - built in kitchen appliances: True
Feature - security: True
Feature - covered parking: True
Feature - pets allowed: True
Feature - pool: True
Feature - electricity meter: True
Feature - elevator: True


In [40]:
for key, value in page_data.items():
    print(f'{key} : {value}')

title : Apartment for sale 160M prime location lowest price in mountain view i city
price : EGP 6,000,000
listing_date : 01-08-2026
seller_name : Views Real Estate
ownership : Resale
area (m²) : 160
bedrooms : 3
bathrooms : 3
furnished : No
property_subtype : Apartment
level : 1
payment option : cash
completion status : ready
balcony : True
built in kitchen appliances : True
security : True
covered parking : True
pets allowed : True
pool : True
electricity meter : True
elevator : True


##### Now we want to know how to get the data from each page so we inspect the page and start taking notes
##### then I will try to take one property and see how to get its data then I will from there build the logic 
##### but we need first to understand that each filter will change the url and the data we get
##### to get all data we need to loop over all filters and get the data from each page and be aware of dublicates but this problem will be solved later 
##### let's now try to get the data from one property 
#### for example this one will be in: 
1. Cairo as all the data we will be collecting
2. Apartment sale
3. In New Cairo
4. 5th setlment
5. mountain-view-icity-compound<br>
**Lastly All the data will be taken from the property page not like in the generlization some will be taken before entering the property page**

In [41]:
# Know we will have to append the data to the dataframe first for the first instance there are some information that we already know about it 
# After that we will get this information scrapped as well 
page_data['city'] = 'cairo'
page_data['district'] = 'new cairo'
page_data['area'] = 'fifth settlement'
page_data['neighborhood'] = 'mountain view i city compound'
page_data['property_type'] = 'apartment'
page_data['sale/rent'] = 'sale'


In [42]:
features = config['data_source']['realstate_website']['features_list']
for feature in features:
    if feature not in page_data:
        page_data[feature] = False
        print(f"Feature - {feature}: False")

Feature - water meter: False
Feature - natural gas: False
Feature - private garden: False
Feature - landline: False
Feature - maid room: False
Feature - built in kitchen: False
Feature - central a/c & heating: False


In [15]:
# apending the data will have some logic let's start with the simple appending logic 
# first mapping between the column data and the page data keys 
df_cols = df.columns.tolist()
for col in df_cols:
    if col not in page_data:
        page_data[col] = None
        print(f"Column {col} not found in page data, setting to None")

Column delivery time not found in page data, setting to None


In [43]:
rows = []
rows.append(page_data)
df = pd.DataFrame(rows, columns=fields)
df.head()


,title,property_type,sale/rent,city,district,area,neighborhood,property_subtype,ownership,electricity meter,...,furnished,level,bedrooms,bathrooms,payment option,delivery time,price,seller_name,area (m²),listing_date
0,Apartment for sale 160M prime location lowest ...,apartment,sale,cairo,new cairo,fifth settlement,mountain view i city compound,Apartment,Resale,True,...,No,1,3,3,cash,NaN,"EGP 6,000,000",Views Real Estate,160,01-08-2026


In [44]:
df .info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  1 non-null      object 
 1   property_type          1 non-null      object 
 2   sale/rent              1 non-null      object 
 3   city                   1 non-null      object 
 4   district               1 non-null      object 
 5   area                   1 non-null      object 
 6   neighborhood           1 non-null      object 
 7   property_subtype       1 non-null      object 
 8   ownership              1 non-null      object 
 9   electricity meter      1 non-null      bool   
 10  water meter            1 non-null      bool   
 11  covered parking        1 non-null      bool   
 12  natural gas            1 non-null      bool   
 13  security               1 non-null      bool   
 14  pets allowed           1 non-null      bool   
 15  balcony   

### Starting from here we will work on getting the locations data 
1. We will start by getting the districts data then we will move to the areas and neighborhoods data

In [5]:
# The districts names are under a div with class "d36fe94e"
# each listing is is under a <a> tag containing a hyperlink to filter the properties with this district
# then under the a tag there is a span tag containing the district name and other span tag containing the count of properties in this district
districts_div = soup.find('div', class_='d36fe94e')
districts = districts_div.find_all('a')
districts_list = []
for district in districts:
    if len(district.findAll('span')) < 2:
        continue
    link = district['href']
    name = district.find_all('span')[0].text
    count = district.find_all('span')[1].text
    districts_list.append({'link': link, 'name': name, 'count': count})
print(f"Total districts found: {len(districts_list)}")
for d in districts_list:
    print(d)
       

NameError: name 'soup' is not defined

In [8]:
district_slugs = []
for district in districts_list:
    district_slugs.append(district['link'][15:-1])
    print(district_slugs[-1])

new-cairo
madinaty
mostakbal-city
new-capital-city
shorouk-city
nasr-city
katameya
new-heliopolis
maadi
sheraton
zahraa-al-maadi
mokattam
heliopolis
obour-city
badr-city
el-fostat
zamalek
new-nozha
ain-shams
hadayek-al-kobba
hadayeq-el-zeitoun
al-manial
gesr-al-suez
downtown-cairo
almazah
shubra
helmeyat-el-zaytoun
marg
helwan
nozha
salam-city
hadayek-helwan
abasiya
15-may-city
masr-al-kadema
matareya
rod-al-farag
boulaq-abo-el-ela
garden-city
basateen


In [5]:
def get_sub_locations(soup, neighborhood = True, number_of_districts=None, parent_location_name=None):
    # location_div = soup.find('div', class_='d36fe94e')
    # print(location_div.prettify())
    try:
        location_div = soup.find('div', class_='d36fe94e')
        locations = location_div.find_all('a')
    except Exception as e:
        print(f"Error finding location div: {e}")
        return []
    # print(f"Total locations found: {len(locations)}")
    # if len(locations) == number_of_districts:
    #     for location in locations:
    #         if len(location.findAll('span')) < 2:
    #             continue
    #         name = location.find_all('span')[0].text
    #         print(f"Checking if location {name} is the same as parent location {parent_location_name}")
    #         if name == parent_location_name:
    #             print(f"Locations list is the same as parent location {parent_location_name}, has no sub-locations")
    #             return None
           
    locations_list = []

    for location in locations:
        if len(location.findAll('span')) < 2:
            continue
        slug = location['href'][15:-1]
        name = location.find_all('span')[0].text
        count = location.find_all('span')[1].text
        # print(f'working on {name} data')
        if len(locations) >= number_of_districts and name == parent_location_name:
                # print(f"Locations list is the same as parent location {parent_location_name}, has no sub-locations")
                return []
        if neighborhood:
       
            # sub_soup = get_page_data(slug)
            # areas = get_sub_locations(sub_soup, neighborhood=False)
            # if len(areas) == len(locations) or not areas:  # Check if areas is an empty list or None
            #     areas = name
            #     print(f"No areas found for {name}")
            locations_list.append({'slug': slug, 'name': name, 'count': count, 'neighborhoods': None})
            
        else:
            locations_list.append({'slug': slug, 'name': name, 'count': count})
        # print(f'Finished working on {name}')
    return locations_list

In [6]:
mouintain_view_soup = get_page_data('mountain-view-3')


Trying fast approach (requests)...
Response status: 200, length: 6023018
Content looks valid, using requests response
Success with requests!
Data extracted for mountain-view-3


In [13]:
mv_sub_locations = get_sub_locations(
    mouintain_view_soup,
    neighborhood=False,
    number_of_districts=40,
    parent_location_name='Mountain View'
)
print(f"Sub-locations for Mountain View: {mv_sub_locations}")

Sub-locations for Mountain View: [{'slug': 'hyde-park-new-cairo-compound', 'name': 'Hyde Park New Cairo Compound', 'count': '(3440)'}, {'slug': 'mivida-compound', 'name': 'Mivida Compound', 'count': '(3255)'}, {'slug': 'mountain-view-icity-compound', 'name': 'Mountain View iCity Compound', 'count': '(2890)'}, {'slug': 'palm-hills-new-cairo-compound', 'name': 'Palm Hills New Cairo Compound', 'count': '(1906)'}, {'slug': 'fifth-square-compound', 'name': 'Fifth Square Compound', 'count': '(1889)'}, {'slug': 'mountain-view-hyde-park-compound', 'name': 'Mountain View Hyde Park Compound', 'count': '(1317)'}, {'slug': 'eastown-compound', 'name': 'Eastown Compound', 'count': '(1284)'}, {'slug': 'beit-al-watan-6', 'name': 'Beit Al Watan', 'count': '(1215)'}, {'slug': 'city-gate-compound', 'name': 'City Gate Compound', 'count': '(1188)'}, {'slug': 'lake-view-residence-compound', 'name': 'Lake view Residence Compound', 'count': '(1156)'}, {'slug': 'el-patio-oro-compound', 'name': 'EL Patio ORO Co

C:\Users\marwa\AppData\Local\Temp\ipykernel_28164\1727533725.py:24: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  if len(location.findAll('span')) < 2:


In [30]:
sub_locations = get_sub_locations(page_soup)

24


In [22]:
print(so.prettify())

<!DOCTYPE html>
<html>
 <body>
  <script type="text/javascript">
   !function(e,t){"object"==typeof exports&&"object"==typeof module?module.exports=t():"function"==typeof define&&define.amd?define([],t):"object"==typeof exports?exports.Fingerprint=t():e.Fingerprint=t()}(self,(()=>(()=>{"use strict";var e={795:function(e,t,n){var r=this&&this.__assign||function(){return r=Object.assign||function(e){for(var t,n=1,r=arguments.length;n<r;n++)for(var i in t=arguments[n])Object.prototype.hasOwnProperty.call(t,i)&&(e[i]=t[i]);return e},r.apply(this,arguments)},i=this&&this.__awaiter||function(e,t,n,r){return new(n||(n=Promise))((function(i,o){function a(e){try{u(r.next(e))}catch(e){o(e)}}function c(e){try{u(r.throw(e))}catch(e){o(e)}}function u(e){var t;e.done?i(e.value):(t=e.value,t instanceof n?t:new n((function(e){e(t)}))).then(a,c)}u((r=r.apply(e,t||[])).next())}))},o=this&&this.__generator||function(e,t){var n,r,i,o,a={label:0,sent:function(){if(1&i[0])throw i[1];return i[1]},trys:[],ops

In [29]:
if not areas:  # Check if areas is an empty list or None
    print(f"no area found for {s}") 

no area found for mohammed-naguib-axis


In [9]:
# Trying to extract the areas from each the district pages 
try:
    scraper = SmartScraper(config)
    districts_soup = scraper.scrape_multiple_pages(district_slugs)

finally:
    scraper.close()



Page 1/40
Trying fast approach (requests)...
Response status: 200, length: 5563533
Content looks valid, using requests response
Success with requests!

Page 2/40
Trying fast approach (requests)...
Response status: 200, length: 5494115
Content looks valid, using requests response
Success with requests!

Page 3/40
Trying fast approach (requests)...
Response status: 200, length: 5555259
Content looks valid, using requests response
Success with requests!

Page 4/40
Trying fast approach (requests)...
Response status: 200, length: 5549962
Content looks valid, using requests response
Success with requests!

Page 5/40
Trying fast approach (requests)...
Response status: 200, length: 5530519
Content looks valid, using requests response
Success with requests!

Page 6/40
Trying fast approach (requests)...
Response status: 200, length: 5490368
Content looks valid, using requests response
Success with requests!

Page 7/40
Trying fast approach (requests)...
Response status: 200, length: 5560924
Cont

In [14]:
for i, district in enumerate(districts_list):
    district['soup'] = districts_soup[i]

In [15]:
len(districts_soup)

40

In [36]:
len(districts_list)

40

In [16]:
locations = []
number_of_districts = len(districts_list)
for district in districts_list:
    district_slug = district['link'][15:-1]
    district_name = district['name']
    soup2 = district['soup']
    areas_list = get_sub_locations(
        soup2, neighborhood=True, 
        number_of_districts = number_of_districts,  
        parent_location_name=district_name
        )
    
    locations.append({'district': district_name, 'slug': district_slug, 'areas': areas_list})
    print(f'Finished extracting areas for district {district_name}')

   
    


C:\Users\marwa\AppData\Local\Temp\ipykernel_28164\1727533725.py:24: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  if len(location.findAll('span')) < 2:


Finished extracting areas for district New Cairo
Finished extracting areas for district Madinaty
Finished extracting areas for district Mostakbal City
Finished extracting areas for district New Capital City
Finished extracting areas for district Shorouk City
Finished extracting areas for district Nasr City
Finished extracting areas for district Katameya
Finished extracting areas for district New Heliopolis
Finished extracting areas for district Maadi
Finished extracting areas for district Sheraton
Finished extracting areas for district Zahraa Al Maadi
Finished extracting areas for district Mokattam
Finished extracting areas for district Heliopolis
Finished extracting areas for district Obour City
Finished extracting areas for district Badr City
Finished extracting areas for district El Fostat
Finished extracting areas for district Zamalek
Finished extracting areas for district New Nozha
Finished extracting areas for district Ain Shams
Finished extracting areas for district Hadayek al-K

[{'district': 'New Cairo', 'slug': 'new-cairo', 'areas': [{'slug': '5th-settlement', 'name': '5th Settlement', 'count': '(44621)', 'neighborhoods': None}, {'slug': '1st-settlement', 'name': '1st Settlement', 'count': '(5840)', 'neighborhoods': None}, {'slug': 'rehab-city', 'name': 'Rehab City', 'count': '(5145)', 'neighborhoods': None}, {'slug': '6th-settlement', 'name': '6th Settlement', 'count': '(1562)', 'neighborhoods': None}, {'slug': '3rd-settlement', 'name': '3rd Settlement', 'count': '(362)', 'neighborhoods': None}, {'slug': 'hyde-park-central', 'name': 'Hyde Park Central', 'count': '(202)', 'neighborhoods': None}, {'slug': 'ozone-medical-center', 'name': 'Ozone Medical Center', 'count': '(58)', 'neighborhoods': None}, {'slug': 'mohammed-naguib-axis', 'name': 'Mohammed Naguib Axis', 'count': '(53)', 'neighborhoods': None}, {'slug': 'gamal-abd-al-nasser-axis', 'name': 'Gamal Abd Al-Nasser Axis', 'count': '(42)', 'neighborhoods': None}, {'slug': 'dijar', 'name': 'Dijar', 'count':

In [17]:
number_of_areas = 0
for location in locations:
   # print(f"{location['district']} has {len(location['areas'])} areas")
   if len(location['areas']) == 40 or len(location['areas']) == 1:
       print(f"district {location['district']} has {len(location['areas'])} areas")
       print(location['areas'])

   number_of_areas += len(location['areas'])
#    print("--------------------")
print(f"Total number of areas: {number_of_areas}")

district Madinaty has 1 areas
[{'slug': 'privado', 'name': 'Privado', 'count': '(603)', 'neighborhoods': None}]
district New Capital City has 40 areas
[{'slug': 'la-vista-city', 'name': 'La Vista City', 'count': '(702)', 'neighborhoods': None}, {'slug': 'al-maqsad', 'name': 'Al Maqsad', 'count': '(628)', 'neighborhoods': None}, {'slug': 'noor-city-2', 'name': 'Noor City', 'count': '(604)', 'neighborhoods': None}, {'slug': 'r8', 'name': 'R8', 'count': '(289)', 'neighborhoods': None}, {'slug': 'new-garden-city', 'name': 'New Garden City', 'count': '(249)', 'neighborhoods': None}, {'slug': 'il-bosco', 'name': 'IL Bosco', 'count': '(159)', 'neighborhoods': None}, {'slug': 'r3', 'name': 'R3', 'count': '(111)', 'neighborhoods': None}, {'slug': 'celia', 'name': 'Celia', 'count': '(106)', 'neighborhoods': None}, {'slug': 'r4', 'name': 'R4', 'count': '(101)', 'neighborhoods': None}, {'slug': 'r7', 'name': 'R7', 'count': '(97)', 'neighborhoods': None}, {'slug': 'taj-tower', 'name': 'Taj Tower', 

In [7]:
def get_multiple_pages(slugs):
    scrapper = SmartScraper(config)
    try:
        pages_soup = scrapper.scrape_multiple_pages(slugs)
    finally:
        scrapper.close()
    return pages_soup

In [8]:
for i, location in enumerate(locations):
    location_slugs = []
    if len(location['areas']) >0:
        for area in location['areas']:
            location_slugs.append(area['slug'])
    else:
        continue
    if len(location_slugs) == 0:
        print(f"No areas found for district {location['district']}, skipping soup extraction")
        locations[i]['areas_soups'] = []
        continue
    neigbhorhood_soups = get_multiple_pages(location_slugs)
    locations[i]['areas_soups'] = neigbhorhood_soups
    print(f'Finished extracting areas soups for district {location["district"]}')


NameError: name 'locations' is not defined

In [76]:
print(f"{locations[0]['district']}  area 1 is {locations[0]['areas'][0]['name']} and has {len(locations[0]['neighborhood_soups'])} neighborhood soups")

New Cairo  area 1 is 5th Settlement and has 21 neighborhood soups


conditionied stasfied


In [22]:
for location in locations:
    try:
        if len(location['areas_soups']) == 0:
            continue
        for i, so in enumerate(location['areas_soups']):
            if so and location['areas'][i]['name'] != 'Mountain View': # Check if soup and area exist
                # print(f"Processing neighborhood soup for {location['district']} - in area {location['areas'][i]['name']}")
                number_of_areas = len(location['areas'])
                neigbhorhoods = get_sub_locations(
                    so, 
                    neighborhood=False, 
                    number_of_districts=number_of_areas, 
                    parent_location_name=location['areas'][i]['name'])
                location['areas'][i]['neighborhoods'] = neigbhorhoods
                if(len(neigbhorhoods) > 0):
                    print(f"Finished processing neighborhood district {location['district']} in area {location['areas'][i]['name']} with {len(neigbhorhoods)} neighborhoods")
                if len(neigbhorhoods) >= 40:
                    print(f"district {location['district']} in area {location['areas'][i]['name']} first neighboroods is {neigbhorhoods[0]} and last neighborhood is {neigbhorhoods[-1]}")
    except Exception as e:
        print(f"Error processing neighborhood soups for {location['district']}: {e}")

C:\Users\marwa\AppData\Local\Temp\ipykernel_28164\1727533725.py:24: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  if len(location.findAll('span')) < 2:


Finished processing neighborhood district New Cairo in area 5th Settlement with 40 neighborhoods
district New Cairo in area 5th Settlement first neighboroods is {'slug': 'hyde-park-new-cairo-compound', 'name': 'Hyde Park New Cairo Compound', 'count': '(3463)'} and last neighborhood is {'slug': 'south-investors-2', 'name': 'South Investors', 'count': '(230)'}
Finished processing neighborhood district New Cairo in area 1st Settlement with 40 neighborhoods
district New Cairo in area 1st Settlement first neighboroods is {'slug': 'taj-city-compound', 'name': 'Taj City Compound', 'count': '(1665)'} and last neighborhood is {'slug': 'el-masrawia-compound-2', 'name': 'El Masrawia Compound', 'count': '(25)'}
Finished processing neighborhood district New Cairo in area Rehab City with 2 neighborhoods
Finished processing neighborhood district New Cairo in area 6th Settlement with 17 neighborhoods
Finished processing neighborhood district New Cairo in area 3rd Settlement with 3 neighborhoods
Finish

In [23]:
for location in locations:
    # deleting the areas soup safely
    location.pop('areas_soups', None)

In [9]:
import yaml

# Load
locs = config['locations']

# Rebuild each district with correct key order
for district in locs:
    # Create new ordered dict
    ordered = {
        'district': district['district'],
        'slug': district['slug'],
        'areas': district.get('areas', [])
    }
    
    # Also fix area order
    for area in ordered['areas']:
        # Reorder area keys in place
        temp = {
            'name': area['name'],
            'slug': area['slug'],
            'count': area.get('count'),
            'neighborhoods': area.get('neighborhoods', [])
        }
        area.clear()
        area.update(temp)
    
    # Update original dict
    district.clear()
    district.update(ordered)

# Save back
# with open('locations.yaml', 'w') as f:
#     yaml.dump(data, f, default_flow_style=False, sort_keys=False, indent=2)

# print("✅ Fixed!")

In [10]:
print("Data being written:")
print(yaml.dump({'locations': locs}, default_flow_style=False, sort_keys=False, indent=2, allow_unicode=True))

Data being written:
locations:
- district: New Cairo
  slug: new-cairo
  areas:
  - name: 5th Settlement
    slug: 5th-settlement
    count: (43805)
    neighborhoods:
    - count: (3463)
      name: Hyde Park New Cairo Compound
      slug: hyde-park-new-cairo-compound
    - count: (3280)
      name: Mivida Compound
      slug: mivida-compound
    - count: (2916)
      name: Mountain View iCity Compound
      slug: mountain-view-icity-compound
    - count: (1925)
      name: Palm Hills New Cairo Compound
      slug: palm-hills-new-cairo-compound
    - count: (1907)
      name: Fifth Square Compound
      slug: fifth-square-compound
    - count: (1323)
      name: Mountain View Hyde Park Compound
      slug: mountain-view-hyde-park-compound
    - count: (1293)
      name: Eastown Compound
      slug: eastown-compound
    - count: (1226)
      name: Beit Al Watan
      slug: beit-al-watan-6
    - count: (1192)
      name: City Gate Compound
      slug: city-gate-compound
    - count: (11

In [11]:
len(locs)

40

In [12]:
# Write the location in config file 
config['locations'] = locs



In [13]:
# Updating the config file with the new locations data 
# Write back to file
with open('../config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False, indent=2, allow_unicode=True)

In [29]:
for location in locations:
    print(location)
    break

{'district': 'New Cairo', 'slug': 'new-cairo', 'areas': [{'slug': '5th-settlement', 'name': '5th Settlement', 'count': '(43805)', 'neighborhoods': [{'slug': 'hyde-park-new-cairo-compound', 'name': 'Hyde Park New Cairo Compound', 'count': '(3463)'}, {'slug': 'mivida-compound', 'name': 'Mivida Compound', 'count': '(3280)'}, {'slug': 'mountain-view-icity-compound', 'name': 'Mountain View iCity Compound', 'count': '(2916)'}, {'slug': 'palm-hills-new-cairo-compound', 'name': 'Palm Hills New Cairo Compound', 'count': '(1925)'}, {'slug': 'fifth-square-compound', 'name': 'Fifth Square Compound', 'count': '(1907)'}, {'slug': 'mountain-view-hyde-park-compound', 'name': 'Mountain View Hyde Park Compound', 'count': '(1323)'}, {'slug': 'eastown-compound', 'name': 'Eastown Compound', 'count': '(1293)'}, {'slug': 'beit-al-watan-6', 'name': 'Beit Al Watan', 'count': '(1226)'}, {'slug': 'city-gate-compound', 'name': 'City Gate Compound', 'count': '(1192)'}, {'slug': 'lake-view-residence-compound', 'nam

In [30]:
print(locations[0].keys())

dict_keys(['district', 'slug', 'areas'])


In [31]:
print("Data being written:")
print(yaml.dump({'locations': locations}, default_flow_style=False, allow_unicode=True))

Data being written:
locations:
- areas:
  - count: (43805)
    name: 5th Settlement
    neighborhoods:
    - count: (3463)
      name: Hyde Park New Cairo Compound
      slug: hyde-park-new-cairo-compound
    - count: (3280)
      name: Mivida Compound
      slug: mivida-compound
    - count: (2916)
      name: Mountain View iCity Compound
      slug: mountain-view-icity-compound
    - count: (1925)
      name: Palm Hills New Cairo Compound
      slug: palm-hills-new-cairo-compound
    - count: (1907)
      name: Fifth Square Compound
      slug: fifth-square-compound
    - count: (1323)
      name: Mountain View Hyde Park Compound
      slug: mountain-view-hyde-park-compound
    - count: (1293)
      name: Eastown Compound
      slug: eastown-compound
    - count: (1226)
      name: Beit Al Watan
      slug: beit-al-watan-6
    - count: (1192)
      name: City Gate Compound
      slug: city-gate-compound
    - count: (1162)
      name: Lake view Residence Compound
      slug: lake-vie

In [32]:
# Check the actual values
print(f"District value: {locations[0]['district']}")
print(f"Slug value: {locations[0]['slug']}")
print(f"Full first location: {locations[0]}")

District value: New Cairo
Slug value: new-cairo
Full first location: {'district': 'New Cairo', 'slug': 'new-cairo', 'areas': [{'slug': '5th-settlement', 'name': '5th Settlement', 'count': '(43805)', 'neighborhoods': [{'slug': 'hyde-park-new-cairo-compound', 'name': 'Hyde Park New Cairo Compound', 'count': '(3463)'}, {'slug': 'mivida-compound', 'name': 'Mivida Compound', 'count': '(3280)'}, {'slug': 'mountain-view-icity-compound', 'name': 'Mountain View iCity Compound', 'count': '(2916)'}, {'slug': 'palm-hills-new-cairo-compound', 'name': 'Palm Hills New Cairo Compound', 'count': '(1925)'}, {'slug': 'fifth-square-compound', 'name': 'Fifth Square Compound', 'count': '(1907)'}, {'slug': 'mountain-view-hyde-park-compound', 'name': 'Mountain View Hyde Park Compound', 'count': '(1323)'}, {'slug': 'eastown-compound', 'name': 'Eastown Compound', 'count': '(1293)'}, {'slug': 'beit-al-watan-6', 'name': 'Beit Al Watan', 'count': '(1226)'}, {'slug': 'city-gate-compound', 'name': 'City Gate Compoun

{locations: [{areas: [{count: (603), name: Privado, neighborhoods: [], slug: privado}],
      district: Madinaty, slug: madinaty}]}



In [38]:
def restructure_locations(flat_locations):
    """
    Restructure to have 'district' as a wrapper key
    """
    districts = {}
    
    for location in flat_locations:
        district_slug = location.get('slug')
        district_name = location.get('district')
        
        # Create district if doesn't exist
        if district_slug not in districts:
            districts[district_slug] = {
                'district': {  # ← 'district' as wrapper
                    'name': district_name,
                    'slug': district_slug,
                    'areas': []
                }
            }
        
        # Add areas to this district
        for area in location.get('areas', []):
            districts[district_slug]['district']['areas'].append({
                'name': area.get('name'),
                'slug': area.get('slug'),
                'count': area.get('count'),
                'neighborhoods': area.get('neighborhoods', [])
            })
    
    # Convert dict to list
    return [v for v in districts.values()]

# Usage
structured_data = restructure_locations(locations)

In [44]:
locations[locations.index(next(loc for loc in locations if loc['district'] == 'Zamelk'))]['areas']

StopIteration: 

In [8]:
locations = config['locations']
loc = locations[1:3]
print("Data being written:")
print(yaml.dump({'locations': loc}, default_flow_style=False, allow_unicode=True, sort_keys=False))

Data being written:
locations:
- areas:
  - count: (603)
    name: Privado
    neighborhoods: []
    slug: privado
  district: Madinaty
  slug: madinaty
- areas:
  - count: (2475)
    name: Sarai
    neighborhoods: []
    slug: sarai
  - count: (841)
    name: Bloomfields
    neighborhoods: []
    slug: bloomfields
  - count: (365)
    name: L’Avenir
    neighborhoods: []
    slug: l-avenir
  - count: (215)
    name: Haptown
    neighborhoods: []
    slug: haptown
  - count: (200)
    name: Monte Napoleon
    neighborhoods: []
    slug: monte-napoleon
  - count: (199)
    name: Aliva mostakbal city
    neighborhoods: []
    slug: aliva-mostakbal-city
  - count: (196)
    name: Green Square
    neighborhoods: []
    slug: green-square
  - count: (186)
    name: Capital Gardens
    neighborhoods: []
    slug: capital-gardens
  - count: (166)
    name: The Butterfly
    neighborhoods: []
    slug: the-butterfly
  - count: (139)
    name: Park Central
    neighborhoods: []
    slug: park-c

In [ ]:
import pickle 
with open('../locations_data.pkl', 'wb') as f:
    pickle.dump(locations, f)   

KeyboardInterrupt: 

In [ ]:
number_of_areas = 0
for location in locations:
   # print(f"{location['district']} has {len(location['areas'])} areas")
   print(f"district {location['district']} has {len(location['areas'])} areas")
   if len(location['areas']) == 40:
       
       print(location['areas'])

   number_of_areas += len(location['areas'])
#    print("--------------------")
print(f"Total number of areas: {number_of_areas}")

district New Cairo has 21 areas
district Madinaty has 1 areas
district Mostakbal City has 38 areas
district New Capital City has 40 areas
[{'slug': 'la-vista-city', 'name': 'La Vista City', 'count': '(735)', 'neighborhoods': None}, {'slug': 'al-maqsad', 'name': 'Al Maqsad', 'count': '(660)', 'neighborhoods': None}, {'slug': 'noor-city-2', 'name': 'Noor City', 'count': '(618)', 'neighborhoods': None}, {'slug': 'r8', 'name': 'R8', 'count': '(310)', 'neighborhoods': None}, {'slug': 'new-garden-city', 'name': 'New Garden City', 'count': '(243)', 'neighborhoods': None}, {'slug': 'il-bosco', 'name': 'IL Bosco', 'count': '(167)', 'neighborhoods': None}, {'slug': 'r3', 'name': 'R3', 'count': '(113)', 'neighborhoods': None}, {'slug': 'celia', 'name': 'Celia', 'count': '(108)', 'neighborhoods': None}, {'slug': 'r7', 'name': 'R7', 'count': '(96)', 'neighborhoods': None}, {'slug': 'r4', 'name': 'R4', 'count': '(91)', 'neighborhoods': None}, {'slug': 'taj-tower', 'name': 'Taj Tower', 'count': '(87)

In [54]:
for location in locations:
    print(f'{location["district"]} have {len(location["neighborhoods"])} neighborhoods')

KeyError: 'neighborhoods'

In [26]:
l = [{'slug': 'hyde-park-central', 'name': 'Hyde Park Central', 'count': '(193)'}, {'slug': '5th-settlement', 'name': '5th Settlement', 'count': '(43625)'}, {'slug': '1st-settlement', 'name': '1st Settlement', 'count': '(5744)'}, {'slug': 'rehab-city', 'name': 'Rehab City', 'count': '(5013)'}, {'slug': '6th-settlement', 'name': '6th Settlement', 'count': '(1598)'}, {'slug': '3rd-settlement', 'name': '3rd Settlement', 'count': '(371)'}, {'slug': 'ozone-medical-center', 'name': 'Ozone Medical Center', 'count': '(59)'}, {'slug': 'mohammed-naguib-axis', 'name': 'Mohammed Naguib Axis', 'count': '(55)'}, {'slug': 'gamal-abd-al-nasser-axis', 'name': 'Gamal Abd Al-Nasser Axis', 'count': '(41)'}, {'slug': 'el-patio-vida', 'name': 'El Patio Vida', 'count': '(27)'}, {'slug': 'mall-top-90', 'name': 'Mall Top 90', 'count': '(26)'}, {'slug': 'town-residence', 'name': 'Town Residence', 'count': '(21)'}, {'slug': 'dijar', 'name': 'Dijar', 'count': '(18)'}, {'slug': 'top-view', 'name': 'Top view', 'count': '(14)'}, {'slug': 'w-signature', 'name': 'W Signature', 'count': '(13)'}, {'slug': 'lotus-compound', 'name': 'Lotus Compound', 'count': '(10)'}, {'slug': 'loaloa', 'name': 'Loaloa', 'count': '(10)'}, {'slug': 'the-mornings', 'name': 'The Mornings', 'count': '(9)'}, {'slug': 'green-house', 'name': 'Green House', 'count': '(4)'}, {'slug': 'prk-vie', 'name': 'Prk Vie', 'count': '(4)'}, {'slug': 'kemet', 'name': 'Kemet', 'count': '(3)'}]
print(f"Total neighborhoods found: {len(l)}")

Total neighborhoods found: 21


In [11]:
print(districts_list)

[{'link': '/en/properties/new-cairo/', 'name': 'New Cairo', 'count': '(57478)'}, {'link': '/en/properties/madinaty/', 'name': 'Madinaty', 'count': '(11933)'}, {'link': '/en/properties/mostakbal-city/', 'name': 'Mostakbal City', 'count': '(5854)'}, {'link': '/en/properties/new-capital-city/', 'name': 'New Capital City', 'count': '(5108)'}, {'link': '/en/properties/shorouk-city/', 'name': 'Shorouk City', 'count': '(4491)'}, {'link': '/en/properties/nasr-city/', 'name': 'Nasr City', 'count': '(2756)'}, {'link': '/en/properties/katameya/', 'name': 'Katameya', 'count': '(1867)'}, {'link': '/en/properties/new-heliopolis/', 'name': 'New Heliopolis', 'count': '(1751)'}, {'link': '/en/properties/sheraton/', 'name': 'Sheraton', 'count': '(1559)'}, {'link': '/en/properties/maadi/', 'name': 'Maadi', 'count': '(1557)'}, {'link': '/en/properties/mokattam/', 'name': 'Mokattam', 'count': '(1426)'}, {'link': '/en/properties/zahraa-al-maadi/', 'name': 'Zahraa Al Maadi', 'count': '(1418)'}, {'link': '/en

In [9]:
try:
    import lxml
    print("✅ lxml imported successfully!")
    print("Version:", lxml.__version__)
except ImportError as e:
    print("❌ Error:", e)

✅ lxml imported successfully!
Version: 6.0.2


In [12]:
import subprocess

# Use uv to install instead of pip
result = subprocess.run(
    ['uv', 'pip', 'install', '--reinstall', 'lxml'],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


Using Python 3.9.2 environment at: D:\Marwan\Projects\cairo_real_state_intelligence_platform\.venv
Resolved 1 package in 651ms
Prepared 1 package in 0.83ms
Uninstalled 1 package in 27ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 93ms
 ~ lxml==6.0.2



### This section is the basic logic implementation for how the whole the website will be scrapped 
we have 40 districts each district may have more than one area and each area may have more than one neighborhood. First there is multiple problem I have noticed when viewing the website we scrape the data in some areas that have neighborhoods there are properties that are listed at the area level but not assigned to any of the neighborhoods. this happens in the areas that have 1 to 4 neighborhoods and this also happens when the districts doesn't have alot of areas


In [7]:
locations = config['locations']
locations[0]['areas'][0]['neighborhoods']

[{'count': '(3463)',
  'name': 'Hyde Park New Cairo Compound',
  'slug': 'hyde-park-new-cairo-compound'},
 {'count': '(3280)', 'name': 'Mivida Compound', 'slug': 'mivida-compound'},
 {'count': '(2916)',
  'name': 'Mountain View iCity Compound',
  'slug': 'mountain-view-icity-compound'},
 {'count': '(1925)',
  'name': 'Palm Hills New Cairo Compound',
  'slug': 'palm-hills-new-cairo-compound'},
 {'count': '(1907)',
  'name': 'Fifth Square Compound',
  'slug': 'fifth-square-compound'},
 {'count': '(1323)',
  'name': 'Mountain View Hyde Park Compound',
  'slug': 'mountain-view-hyde-park-compound'},
 {'count': '(1293)', 'name': 'Eastown Compound', 'slug': 'eastown-compound'},
 {'count': '(1226)', 'name': 'Beit Al Watan', 'slug': 'beit-al-watan-6'},
 {'count': '(1192)',
  'name': 'City Gate Compound',
  'slug': 'city-gate-compound'},
 {'count': '(1162)',
  'name': 'Lake view Residence Compound',
  'slug': 'lake-view-residence-compound'},
 {'count': '(1100)',
  'name': 'EL Patio ORO Compound'

In [5]:
# let's start building the code
locations = config['locations']
empty_pd = []

for property_type in config['data_source']['realstate_website']['property_types']:
        no_dis = 0
        parts = property_type.split('-')
        type_ = parts[0]
        sale_or_rent = parts[-1]
        logs = []
        for location in locations:
            district_slug = location['slug']
            if len(location['areas']) == 0:
                url = f"{config['data_source']['realstate_website']['base_url']}{property_type}/{district_slug}/"
                empty_pd.append({'property_type': type_, 'sale/rent': sale_or_rent, 'city': 'Cairo', 'district': location['district'], 'area': None, 'neighborhood': None, 'link': url})
                logs.append(url)
                continue
            elif len(location['areas']) < 3:
                no_dis += 1
                url = f"{config['data_source']['realstate_website']['base_url']}{property_type}/{district_slug}/"
                empty_pd.append({'property_type': type_, 'sale/rent': sale_or_rent, 'city': 'Cairo', 'district': location['district'], 'area': None, 'neighborhood': None, 'link': url})
                logs.append(url)
            for area in location['areas']:
               
                area_slug = area['slug']
                
                     
                if len(area['neighborhoods']) == 0:
                    url = f"{config['data_source']['realstate_website']['base_url']}{property_type}/{area_slug}/"
                    empty_pd.append({'property_type': type_, 'sale/rent': sale_or_rent, 'city': 'Cairo', 'district': location['district'], 'area': area['name'], 'neighborhood': None, 'link': url})
                    logs.append(url)
                    continue
             
                 
                for neighborhood in area['neighborhoods']:
                    neighborhood_slug = neighborhood['slug']
                    url = f"{config['data_source']['realstate_website']['base_url']}{property_type}/{neighborhood_slug}/"
                    empty_pd.append({'property_type': type_, 'sale/rent': sale_or_rent, 'city': 'Cairo', 'district': location['district'], 'area': area['name'], 'neighborhood': neighborhood['name'], 'link': url})
                    logs.append(url)
            # print(f"Finished constructing URLs for district {location['district']} with {len(logs)} URLS")
        print(f"Total URLs constructed for {property_type}: {len(logs)}")
        print(f"district with less than 3 areas: {no_dis}")

Total URLs constructed for apartments-duplex-for-sale: 456
district with less than 3 areas: 13
Total URLs constructed for apartments-duplex-for-rent: 456
district with less than 3 areas: 13
Total URLs constructed for villas-for-sale: 456
district with less than 3 areas: 13
Total URLs constructed for villas-for-rent: 456
district with less than 3 areas: 13


In [6]:
base_link_df = pd.DataFrame(empty_pd)


In [11]:
base_link_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1824 entries, 0 to 1823
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   property_type  1824 non-null   object
 1   sale/rent      1824 non-null   object
 2   city           1824 non-null   object
 3   district       1824 non-null   object
 4   area           1732 non-null   object
 5   neighborhood   488 non-null    object
 6   link           1824 non-null   object
dtypes: object(7)
memory usage: 99.9+ KB


In [7]:
base_link_df.head()

,property_type,sale/rent,city,district,area,neighborhood,link
0,apartments,sale,Cairo,New Cairo,5th Settlement,Hyde Park New Cairo Compound,https://www.dubizzle.com.eg/en/properties/apar...
1,apartments,sale,Cairo,New Cairo,5th Settlement,Mivida Compound,https://www.dubizzle.com.eg/en/properties/apar...
2,apartments,sale,Cairo,New Cairo,5th Settlement,Mountain View iCity Compound,https://www.dubizzle.com.eg/en/properties/apar...
3,apartments,sale,Cairo,New Cairo,5th Settlement,Palm Hills New Cairo Compound,https://www.dubizzle.com.eg/en/properties/apar...
4,apartments,sale,Cairo,New Cairo,5th Settlement,Fifth Square Compound,https://www.dubizzle.com.eg/en/properties/apar...


In [8]:
base_link_df.describe()

,property_type,sale/rent,city,district,area,neighborhood,link
count,1772,1772,1772,1772,1732,488,1772
unique,2,2,1,40,318,122,1772
top,apartments,sale,Cairo,New Cairo,1st Settlement,Hyde Park New Cairo Compound,https://www.dubizzle.com.eg/en/properties/vill...
freq,886,886,1772,472,160,4,1


In [10]:
base_link_df['link'][0]

'https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/hyde-park-new-cairo-compound/'

In [23]:
test2_main_page = base_link_df['link'][0]

In [24]:
tagmo3_test_page = get_page_data(test2_main_page)

Trying fast approach (requests)...
Response status: 200, length: 5165256
Content looks valid, using requests response
Success with requests!
Data extracted for https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/hyde-park-new-cairo-compound/


In [13]:
test_main_page = 'https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/al-manial/'
manial_test_page = get_page_data(test_main_page)


Trying fast approach (requests)...
Response status: 200, length: 5036331
Content looks valid, using requests response
Success with requests!
Data extracted for https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/al-manial/


In [25]:
properties_list_div = tagmo3_test_page.select_one(config['data_source']['realstate_website']['main_page_selectors']['properties_div'])

In [26]:
properties_unordered_lists = properties_list_div.find_all('ul', recursive=False)


In [28]:
number_of_properties = 0
for ul in properties_unordered_lists:
    next_div = ul.find_next('div')
    listed_items = ul.find_all('li')
    next_div_text = next_div.span.text if next_div and next_div.span else "No class found"
    if next_div_text == 'Results from other locations':
        print("Reached the end of listings for this location, stopping extraction")
        break
    number_of_properties += len(listed_items)
    for item in listed_items:
        try:
            link = item.find('a')['href']
            # print(link)
        except Exception as e:
            print(f"This div doesn't contain a listing")
print(f"Total number of properties found: {number_of_properties}")

This div doesn't contain a listing
Total number of properties found: 46


In [34]:
tagmo3_df = base_link_df[base_link_df['district'] == 'New Cairo']

In [35]:
tagmo3_df.info() 

<class 'pandas.core.frame.DataFrame'>
Index: 472 entries, 0 to 1446
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   property_type  472 non-null    object
 1   sale/rent      472 non-null    object
 2   city           472 non-null    object
 3   district       472 non-null    object
 4   area           472 non-null    object
 5   neighborhood   408 non-null    object
 6   link           472 non-null    object
dtypes: object(7)
memory usage: 29.5+ KB


### I'm now in the phase of I have each location URL and the scrapping is going to go through this phases:

1. I have 2 property types (apartment and villa) and 2 sale/rent types (sale and rent) so I will loop through each combination of these types and for each combination I will loop through the locations and construct the URL for each location.
   
2. Each location have more the one property maybe in one page or more in the first phase I want to extract the links for each property and save it in the dataframe with the already known information about this property.
   
3. Phase one is completed after i have links for all of the properties then I will go for each link and extract the property actuall data.
   

{'title': None, 'property_type': None, 'sale/rent': None, 'city': None, 'district': None, 'area': None, 'neighborhood': None, 'property_subtype': None, 'ownership': None, 'electricity meter': None, 'water meter': None, 'covered parking': None, 'natural gas': None, 'security': None, 'pets allowed': None, 'balcony': None, 'private garden': None, 'pool': None, 'landline': None, 'maid room': None, 'built in kitchen': None, 'elevator': None, 'central a/c & heating': None, 'furnished': None, 'level': None, 'bedrooms': None, 'bathrooms': None, 'payment option': None, 'delivery time': None, 'price': None, 'seller_name': None, 'area (mÂ²)': None, 'link': None}


In [7]:
def extract_properties_links(main_page_url, property_schema):
    try:
        page_soup = get_page_data(main_page_url)    
        properties_list_div = page_soup.select_one(config['data_source']['realstate_website']['main_page_selectors']['properties_div'])
    
        stop_pagenation_selector = page_soup.select_one(config['data_source']['realstate_website']['selectors']['stop_pagenation'])
        if stop_pagenation_selector and "Unfortunately we haven't found anything" in stop_pagenation_selector.text:
            print(f"No properties div found for {main_page_url}, skipping extraction")
            return []
        
        properties_unordered_lists = properties_list_div.find_all('ul', recursive=False)
        links = []
        for ul in properties_unordered_lists:
            next_div = ul.find_next('div')
            listed_items = ul.find_all('li')
            next_div_text = next_div.span.text if next_div and next_div.span else "No class found"
            if next_div_text == 'Results from other locations':
                print("Reached the end of listings for this location, stopping extraction")
                break
            for item in listed_items:
                link_container = item.find('a', href=True)
                
                if not link_container:  # expected case — just skip
                    continue
                
                link = link_container['href']
                title = link_container.get('title', 'No title found')
                
                if not link.startswith('/en/ad'):  # filter non-property links
                    continue
                
                try:
                    # keep try/except only for truly unexpected failures
                    prop = property_schema.copy()
                    prop['link'] = link
                    prop['title'] = title
                    links.append(prop)
                except Exception as e:
                    print(f"Unexpected error processing item: {e}")
                    print(f"Problematic item: {item}")  # print the actual HTML so you can debug
        
        return links
    except Exception as e:      
        print(f"Unexpected error occurred: {e}")
        return []

In [30]:
base_link_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1824 entries, 0 to 1823
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   property_type  1824 non-null   object
 1   sale/rent      1824 non-null   object
 2   city           1824 non-null   object
 3   district       1824 non-null   object
 4   area           1732 non-null   object
 5   neighborhood   488 non-null    object
 6   link           1824 non-null   object
dtypes: object(7)
memory usage: 99.9+ KB


In [8]:
def property_coordinator(df, property_schema):
    all_properties = []
    try:
        
        for location in df.itertuples():
            max_number_of_pages = config['data_source']['realstate_website']['selectors']['max_page_number']
            for i in range(1, max_number_of_pages):
                url =location.link + f"?page={i}"
                properties = extract_properties_links(url, property_schema)
                if not properties:
                    print(f"No properties found on page {url}")
                    break

                for prop in properties:
                    prop['property_type'] = location.property_type
                    prop['sale/rent'] = location._2
                    prop['city'] = location.city
                    prop['district'] = location.district
                    prop['area'] = location.area
                    prop['neighborhood'] = location.neighborhood
                all_properties.extend(properties)

            
        return pd.DataFrame(all_properties)
    except Exception as e:
        print(f"Error in property coordinator: {e}")
        return pd.DataFrame(all_properties)

In [19]:
x = base_link_df['link'].unique()
len(x)
for l in x:
    if 'true' in l:
        print(l)

https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/true
https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-rent/true
https://www.dubizzle.com.eg/en/properties/villas-for-sale/true
https://www.dubizzle.com.eg/en/properties/villas-for-rent/true


In [9]:
fields = config['data_source']['realstate_website']['fields']
fields.append('link')
property_schema = {field: None for field in fields}
properties_df = property_coordinator(base_link_df, property_schema)

Trying fast approach (requests)...
Response status: 200, length: 5526299
Content looks valid, using requests response
Success with requests!
Data extracted for https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/hyde-park-new-cairo-compound/?page=1
Trying fast approach (requests)...
Response status: 200, length: 5515421
Content looks valid, using requests response
Success with requests!
Data extracted for https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/hyde-park-new-cairo-compound/?page=2
Trying fast approach (requests)...
Response status: 200, length: 5505239
Content looks valid, using requests response
Success with requests!
Data extracted for https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/hyde-park-new-cairo-compound/?page=3
Trying fast approach (requests)...
Response status: 200, length: 5506975
Content looks valid, using requests response
Success with requests!
Data extracted for https://www.dubizzle.com.eg/en/properties/

In [10]:
properties_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70565 entries, 0 to 70564
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   title                  70565 non-null  object
 1   property_type          70565 non-null  object
 2   sale/rent              70565 non-null  object
 3   city                   70565 non-null  object
 4   district               70565 non-null  object
 5   area                   63968 non-null  object
 6   neighborhood           40720 non-null  object
 7   property_subtype       0 non-null      object
 8   ownership              0 non-null      object
 9   electricity meter      0 non-null      object
 10  water meter            0 non-null      object
 11  covered parking        0 non-null      object
 12  natural gas            0 non-null      object
 13  security               0 non-null      object
 14  pets allowed           0 non-null      object
 15  balcony            

In [14]:
properties_df.to_csv('../data/properties_version1.csv', index=False)

In [ ]:
# locations = config['locations']
# property_types = config['data_source']['realstate_website']['property_types']
# for property_type in property_types:
#     base_link = config['data_source']['realstate_website']['base_url'] + property_type + '/'
#     urls = []
#     for location in locations:
#         if len(location['areas']) > 0:
#             for area in location['areas']:
#                 if len(area['neighborhoods']) > 0:
#                     for neighborhood in area['neighborhoods']:
#                         # Here for the locations that have neigbhorhoods
#                         url = f"{base_link}{neighborhood['slug']}/"
#                         coordinate_extractor(url, {
#                             'property_type': property_type.split('-')[0],
#                             'sale_or_rent': property_type.split('-')[-1],
#                             'city': 'Cairo',
#                             'district': location['district'],
#                             'area': area['name'],
#                             'neighborhood': neighborhood['name']
#                         })

#                 else:
#                     # Here for that are one the area level
#                     url = f"{base_link}{area['slug']}/"
#                     coordinate_extractor(url, {
#                         'property_type': property_type.split('-')[0],
#                         'sale_or_rent': property_type.split('-')[-1],
#                         'city': 'Cairo',
#                         'district': location['district'],
#                         'area': area['name'],
#                         'neighborhood': None
#                     })

#         else:
#             # Herer for the case of locations that only have a district
#             url = f"{base_link}{location['slug']}/"
#             coordinate_extractor(url, {
#                     'property_type': property_type.split('-')[0],
#                     'sale_or_rent': property_type.split('-')[-1],
#                     'city': 'Cairo',
#                     'district': location['district'],
#                     'area': None,
#                     'neighborhood': None
#                 })
    
          


https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/hyde-park-new-cairo-compound/
https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/alba-aliyah-uptown-cairo/
https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/saint-teresa-el-sahel/
https://www.dubizzle.com.eg/en/properties/apartments-duplex-for-sale/6-october-st-kafr-abu-seer/


In [22]:
locations = config['locations']
num1 = 0
num2 = 0
for location in locations:
    if len(location['areas']) == 0:
        num1 += 1
    for area in location['areas']:
        if len(area['neighborhoods']) == 0:
            num2 += 1
print(f"Number of districts with no areas: {num1}")
print(f"Number of areas with no neighborhoods: {num2}")

Number of districts with no areas: 10
Number of areas with no neighborhoods: 311
